# Notebook 07 — Human Evaluation Analysis

**Project:** MSc AI Dissertation — AI-Generated Text Detection  
**Student:** Abdul Hannaan Mohammed | B00409227 | UWS  
**Week:** 4 (prep) / 8–9 (analysis)

**This notebook has two stages:**

**STAGE 1 (Run now — Week 4):** Select and export text samples for your Google Form.

**STAGE 2 (Run after Week 9):** Load Google Forms responses and produce human evaluation results.

**Study design:**  
13–14 participants each read 15 text samples and judge each as *Human-written* or *AI-generated*.  
Samples are drawn from 3 conditions (5 each):
- **Condition A:** Human-written text (from HC3 test set)
- **Condition B:** Original AI-generated text (before rewriting)
- **Condition C:** Rewritten AI text (after paraphrase attack)

> **DISSERTATION NOTE:** Human evaluation results go into Chapter 5 (Results) and Chapter 6 (Discussion).

---
# STAGE 1 — Prepare Samples for Google Form
## Run this now (Week 4) to generate your Google Form content
---

## Cell 1 — Imports and Paths

In [1]:
import os
import json
import random
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 150
import seaborn as sns
sns.set_theme(style='whitegrid', palette='muted')

NOTEBOOK_DIR    = os.path.dirname(os.path.abspath('__file__'))
PROJECT_ROOT    = os.path.dirname(NOTEBOOK_DIR)
DATA_PROCESSED  = os.path.join(PROJECT_ROOT, 'data', 'processed')
DATA_ADV        = os.path.join(PROJECT_ROOT, 'data', 'adversarial')
RESULTS_FIGS    = os.path.join(PROJECT_ROOT, 'results', 'figures')
RESULTS_TABLES  = os.path.join(PROJECT_ROOT, 'results', 'tables')
RESULTS_METRICS = os.path.join(PROJECT_ROOT, 'results', 'metrics')
HE_DIR          = os.path.join(PROJECT_ROOT, 'human_evaluation')

for d in [os.path.join(HE_DIR, 'form_screenshots'),
          os.path.join(HE_DIR, 'responses'),
          os.path.join(HE_DIR, 'analysis')]:
    os.makedirs(d, exist_ok=True)

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print('Libraries loaded.')

Libraries loaded.


## Cell 2 — Select 15 Samples for Google Form

We select:
- **5 human-written** samples from the test set (label=0)
- **5 original AI** samples (label=1, before rewriting)
- **5 rewritten AI** samples (after paraphrase attack)

All samples are truncated to 150 words to keep the form readable.

In [2]:
MAX_WORDS_FOR_FORM = 150   # keep samples short so participants can read quickly

def truncate(text, max_words=MAX_WORDS_FOR_FORM):
    words = str(text).split()
    if len(words) <= max_words:
        return text
    return ' '.join(words[:max_words]) + '...'


# ── Load test set ──────────────────────────────────────────────────────────────
test_df = pd.read_csv(os.path.join(DATA_PROCESSED, 'test.csv'))

# 5 human samples
human_samples = (
    test_df[test_df['label'] == 0]
    .sample(5, random_state=RANDOM_SEED)['text']
    .apply(truncate)
    .tolist()
)

# ── Load adversarial set ───────────────────────────────────────────────────────
adv_df = pd.read_csv(os.path.join(DATA_ADV, 'pegasus_rewritten_500.csv'))
adv_df = adv_df[adv_df['success'] == True].reset_index(drop=True)

# 5 original AI samples (from the adversarial CSV — before rewriting)
sampled_adv = adv_df.sample(5, random_state=RANDOM_SEED)
original_ai_samples = sampled_adv['original_text'].apply(truncate).tolist()

# 5 rewritten AI samples (after rewriting) — same rows as above
rewritten_samples = sampled_adv['rewritten_text'].apply(truncate).tolist()


# ── Build the form sample list ─────────────────────────────────────────────────
form_samples = []
for i, text in enumerate(human_samples):
    form_samples.append({'sample_id': f'H{i+1}', 'condition': 'Human',        'text': text, 'true_label': 'Human'})
for i, text in enumerate(original_ai_samples):
    form_samples.append({'sample_id': f'A{i+1}', 'condition': 'Original AI',  'text': text, 'true_label': 'AI'})
for i, text in enumerate(rewritten_samples):
    form_samples.append({'sample_id': f'R{i+1}', 'condition': 'Rewritten AI', 'text': text, 'true_label': 'AI'})

# Shuffle so participants see mixed conditions
random.shuffle(form_samples)
for i, s in enumerate(form_samples):
    s['question_number'] = i + 1

form_df = pd.DataFrame(form_samples)

# Save the answer key (do NOT share this with participants)
answer_key_path = os.path.join(HE_DIR, 'analysis', 'answer_key.csv')
form_df.to_csv(answer_key_path, index=False)

print(f'15 samples selected and shuffled.')
print(f'Answer key saved to: {answer_key_path}')
print(f'\nSample distribution:')
print(form_df['condition'].value_counts().to_string())

15 samples selected and shuffled.
Answer key saved to: c:\Users\Abdul\OneDrive\Documents\Uni\MSc\term3\MSc-AI-Detection\human_evaluation\analysis\answer_key.csv

Sample distribution:
condition
Original AI     5
Rewritten AI    5
Human           5


## Cell 3 — Print All 15 Samples for Google Form

Copy each sample text into your Google Form as a question.

> **GOOGLE FORM SETUP INSTRUCTIONS:**
> 1. Go to forms.google.com → create a new blank form
> 2. Title: "AI vs Human Text Detection Study — UWS MSc Research"
> 3. Add a section with: brief instructions, consent statement
> 4. For each of the 15 samples below, add a Multiple Choice question:
>    - Question text: paste the sample text + "Who wrote this text?"
>    - Options: "Human-written" / "AI-generated"
> 5. At the end, add: "How confident were you overall? (1–5 scale)"
> 6. Collect responses via the Responses tab → link to Google Sheets
>
> **SCREENSHOT REMINDER:** Take a screenshot of your completed Google Form.  
> Save as: `screenshots/17_google_form.png`

In [3]:
print('=== SAMPLES TO PASTE INTO GOOGLE FORM ===')
print('Copy each sample exactly as shown below.')
print('The Question Number is what appears in the form (do NOT reveal condition to participants).\n')

for _, row in form_df.sort_values('question_number').iterrows():
    print(f"{'='*70}")
    print(f"QUESTION {row['question_number']} (internal ID: {row['sample_id']}, TRUE: {row['true_label']})")
    print(f"{'='*70}")
    print(f"{row['text']}")
    print(f"\nWho wrote this text?")
    print(f"  ( ) Human-written")
    print(f"  ( ) AI-generated")
    print()

=== SAMPLES TO PASTE INTO GOOGLE FORM ===
Copy each sample exactly as shown below.
The Question Number is what appears in the form (do NOT reveal condition to participants).

QUESTION 1 (internal ID: A4, TRUE: AI)
Childbirth is painful because it is a very physically demanding process. The baby needs to pass through the mother's birth canal, which is a narrow opening in the pelvis. This can be difficult and cause discomfort or pain. Pain during childbirth is a natural response of the body, and it helps to alert the mother that something is happening and to pay attention. Pain during childbirth is not harmful to the baby or the mother. It is just a sign that the baby is being born. In the past, childbirth was more dangerous than it is now because women did not have access to modern medical care. However, even with modern medical care, childbirth can still be uncomfortable.

Who wrote this text?
  ( ) Human-written
  ( ) AI-generated

QUESTION 2 (internal ID: R4, TRUE: AI)
Childbirth is 

---
# STAGE 2 — Analyse Google Forms Responses
## Run this AFTER Week 9 when you have collected all responses
---

**Before running Stage 2:**
1. Export your Google Sheets responses as CSV
2. Save it to: `human_evaluation/responses/responses.csv`
3. Then run Cells 4 onwards

## Cell 4 — Load and Clean Google Forms Responses

In [ ]:
responses_path = os.path.join(HE_DIR, 'responses', 'responses.csv')

# Check if responses file exists
if not os.path.exists(responses_path):
    print('RESPONSES FILE NOT FOUND.')
    print(f'Expected location: {responses_path}')
    print('Export your Google Sheets responses as CSV and save there.')
    print('\nCreating mock data so you can see how the analysis will look...')

    # ── Generate mock data for testing ────────────────────────────────────────
    # Simulates 13 participants responding to 15 questions
    n_participants = 13
    mock_rows = []
    answer_key = form_df.set_index('question_number')['true_label'].to_dict()
    condition_key = form_df.set_index('question_number')['condition'].to_dict()

    for p in range(n_participants):
        row = {'Participant': f'P{p+1:02d}'}
        for q in range(1, 16):
            true = answer_key[q]
            cond = condition_key[q]
            # Human text: participants correct ~70% of time
            # Original AI: participants correct ~65% of time
            # Rewritten AI: participants correct ~45% of time (hard to detect)
            accuracy = {'Human': 0.70, 'Original AI': 0.65, 'Rewritten AI': 0.45}[cond]
            if random.random() < accuracy:
                response = true
            else:
                response = 'Human' if true == 'AI' else 'AI'
            row[f'Q{q}'] = response
        mock_rows.append(row)

    responses_df = pd.DataFrame(mock_rows)
    mock_path = os.path.join(HE_DIR, 'responses', 'responses_mock.csv')
    responses_df.to_csv(mock_path, index=False)
    print(f'Mock data saved to: {mock_path}')
    print('REPLACE WITH REAL DATA when available.')

else:
    responses_df = pd.read_csv(responses_path)
    print(f'Responses loaded: {responses_path}')
    print(f'Participants: {len(responses_df)}')

print(f'\nResponse data shape: {responses_df.shape}')
responses_df.head(3)

## Cell 5 — Compute Human Accuracy by Condition

In [ ]:
# Map question columns to conditions using the answer key
answer_key   = form_df.set_index('question_number')['true_label'].to_dict()
condition_key = form_df.set_index('question_number')['condition'].to_dict()

# Compute per-participant accuracy
results = []
for _, prow in responses_df.iterrows():
    participant = prow.get('Participant', 'Unknown')
    for q in range(1, 16):
        col = f'Q{q}'
        if col not in prow:
            continue
        response  = str(prow[col]).strip()
        true      = answer_key.get(q, '')
        condition = condition_key.get(q, '')
        correct   = 1 if response == true else 0
        results.append({
            'participant': participant,
            'question':    q,
            'condition':   condition,
            'true_label':  true,
            'response':    response,
            'correct':     correct
        })

results_df = pd.DataFrame(results)

# Overall accuracy by condition
condition_acc = results_df.groupby('condition')['correct'].agg(['mean', 'sum', 'count'])
condition_acc.columns = ['Accuracy', 'Correct', 'Total']
condition_acc['Accuracy_%'] = (condition_acc['Accuracy'] * 100).round(1)

print('=== HUMAN EVALUATION RESULTS BY CONDITION ===')
print(condition_acc.to_string())
print()
print(f'Overall human accuracy: {results_df["correct"].mean()*100:.1f}%')
print(f'Total participants    : {responses_df.shape[0]}')
print(f'Total judgements      : {len(results_df)}')

## Cell 6 — Human vs RoBERTa Comparison Chart

> **DISSERTATION NOTE:** This is **Figure 12** in Chapter 5.

In [ ]:
# Load RoBERTa adversarial results
with open(os.path.join(RESULTS_METRICS, 'results_adversarial.json'), 'r') as f:
    adv_results = json.load(f)

roberta_detection_original = adv_results['original_detection_rate'] * 100
roberta_detection_rewritten = adv_results['rewritten_detection_rate'] * 100

# Human accuracy on original AI vs rewritten AI
human_original  = condition_acc.loc['Original AI',  'Accuracy_%'] if 'Original AI'  in condition_acc.index else 0
human_rewritten = condition_acc.loc['Rewritten AI', 'Accuracy_%'] if 'Rewritten AI' in condition_acc.index else 0
human_baseline  = condition_acc.loc['Human',        'Accuracy_%'] if 'Human'        in condition_acc.index else 0

fig, ax = plt.subplots(figsize=(11, 6))

x      = np.arange(2)
width  = 0.35
human_vals   = [human_original,           human_rewritten]
roberta_vals = [roberta_detection_original, roberta_detection_rewritten]

b1 = ax.bar(x - width/2, human_vals,   width, label='Human Participants', color='#4C72B0', edgecolor='white')
b2 = ax.bar(x + width/2, roberta_vals, width, label='RoBERTa Classifier', color='#DD8452', edgecolor='white')

ax.set_title('AI Detection Accuracy: Humans vs RoBERTa\nBefore and After Paraphrase Attack',
             fontsize=13, fontweight='bold')
ax.set_ylabel('Detection Accuracy (%)', fontsize=11)
ax.set_xticks(x)
ax.set_xticklabels(['Original AI Text\n(Before Attack)', 'Rewritten AI Text\n(After Attack)'], fontsize=11)
ax.set_ylim(0, 115)
ax.axhline(y=50, color='gray', linestyle='--', linewidth=1, alpha=0.5, label='Chance (50%)')
ax.legend(fontsize=10)

ax.bar_label(b1, labels=[f'{v:.1f}%' for v in human_vals],   padding=4, fontsize=10)
ax.bar_label(b2, labels=[f'{v:.1f}%' for v in roberta_vals], padding=4, fontsize=10)

plt.tight_layout()

fig_path = os.path.join(RESULTS_FIGS, 'fig12_human_vs_roberta.png')
plt.savefig(fig_path, bbox_inches='tight', dpi=150)
print(f'Figure saved: {fig_path}')
plt.show()

## Cell 7 — Human Accuracy Bar Chart

> **DISSERTATION NOTE:** This is **Figure 13** in Chapter 5.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

conditions = condition_acc.index.tolist()
accuracies = condition_acc['Accuracy_%'].tolist()
colors     = ['#4C72B0', '#DD8452', '#55A868']

bars = ax.bar(conditions, accuracies,
              color=colors[:len(conditions)], edgecolor='white', width=0.5)

ax.axhline(y=50, color='gray', linestyle='--', linewidth=1.5,
           alpha=0.7, label='Chance level (50%)')

ax.set_title('Human Participant Accuracy by Text Condition',
             fontsize=12, fontweight='bold')
ax.set_ylabel('Accuracy (%)', fontsize=11)
ax.set_xlabel('Text Condition', fontsize=11)
ax.set_ylim(0, 110)
ax.legend(fontsize=9)
ax.bar_label(bars, labels=[f'{a:.1f}%' for a in accuracies],
             padding=4, fontsize=11, fontweight='bold')

plt.tight_layout()

fig_path = os.path.join(RESULTS_FIGS, 'fig13_human_accuracy_by_condition.png')
plt.savefig(fig_path, bbox_inches='tight', dpi=150)
print(f'Figure saved: {fig_path}')
plt.show()

## Cell 8 — Save Human Evaluation Results Table

> **DISSERTATION NOTE:** This becomes **Table 6** in Chapter 5.

In [ ]:
human_eval_results = {
    'n_participants'          : int(responses_df.shape[0]),
    'n_questions'             : 15,
    'overall_accuracy'        : round(float(results_df['correct'].mean()), 4),
    'human_text_accuracy'     : round(human_baseline  / 100, 4),
    'original_ai_accuracy'    : round(human_original  / 100, 4),
    'rewritten_ai_accuracy'   : round(human_rewritten / 100, 4),
    'roberta_original_detection': round(roberta_detection_original  / 100, 4),
    'roberta_rewritten_detection': round(roberta_detection_rewritten / 100, 4),
}

he_json_path = os.path.join(RESULTS_METRICS, 'results_human_evaluation.json')
with open(he_json_path, 'w') as f:
    json.dump(human_eval_results, f, indent=2)

# Save table CSV
table_rows = []
for cond in condition_acc.index:
    table_rows.append({
        'Condition'          : cond,
        'Human Accuracy (%)' : condition_acc.loc[cond, 'Accuracy_%'],
        'Correct / Total'    : f"{int(condition_acc.loc[cond,'Correct'])} / {int(condition_acc.loc[cond,'Total'])}",
    })

table_df = pd.DataFrame(table_rows)
table_path = os.path.join(RESULTS_TABLES, 'table06_human_evaluation.csv')
table_df.to_csv(table_path, index=False)

print('=== TABLE 6: Human Evaluation Results ===')
print(table_df.to_string(index=False))
print(f'\nSaved to: {table_path}')
print(json.dumps(human_eval_results, indent=2))

## Notebook Summary

### Stage 1 output (Week 4)
- 15 text samples selected and shuffled for Google Form
- Answer key saved to `human_evaluation/analysis/answer_key.csv`
- Each sample printed above — copy into Google Form

### Stage 2 output (Week 9 — after data collection)
- Human accuracy computed per condition
- Human vs RoBERTa comparison chart produced
- Results table saved for dissertation

### Google Form checklist
- [ ] Create form at forms.google.com
- [ ] Add title: "AI vs Human Text Detection Study — UWS MSc Research"
- [ ] Add consent statement paragraph
- [ ] Add all 15 samples as multiple choice questions (Human-written / AI-generated)
- [ ] Add final confidence rating (1–5 scale)
- [ ] Link responses to Google Sheets
- [ ] Share form with 13–14 participants
- [ ] Screenshot the form → save as `screenshots/17_google_form.png`
- [ ] After collecting responses, export CSV → save to `human_evaluation/responses/responses.csv`
- [ ] Re-run Stage 2 cells (Cells 4–8)